# 『太平寰宇記』（全200巻）歴史GISパイプライン

北宋・太宗の太平興国年間（976–984年）に楽史（がくし）が編纂した地理総志『太平寰宇記』（四庫全書本・全200巻）の漢文テキストから抽出された固有表現（NER）・四至八到里程・山川古跡・戸口データを読み込み、以下の処理を実行する Google Colab ノートブックです。

### パイプライン工程
1. **構造化データ（JSON / CSV）の読み込み**: 十三道・387州府軍監・1,150所属県・3,223四至八到里程・6,934山川古跡・203四夷諸国
2. **CHGIS（ハーバード中国歴史地理情報システム）座標突合 & 宋代1里の実測距離較正**
3. **NetworkX + SciPy による『四至八到』有向距離グラフ構築と非線形最小二乗三辺測量（Trilateration）**
4. **GeoPandas / Shapely による州域 Voronoi ポリゴン・十三道領域の生成と GeoJSON 出力**
5. **Folium（Leaflet）によるインタラクティブ歴史地図の可視化**

In [ ]:
# 必要なライブラリのインストール（Google Colab環境用）
!pip install -q geopandas shapely networkx scipy folium pandas matplotlib

In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
from scipy.optimize import least_squares
import geopandas as gpd
from shapely.geometry import Point, LineString, shape
import folium
from folium.plugins import MarkerCluster, HeatMap

# データディレクトリの設定（Colabにドライブをマウントする場合やローカル実行に対応）
DATA_DIR = Path('../data') if Path('../data').exists() else Path('./data')
GEOJSON_DIR = Path('../geojson') if Path('../geojson').exists() else Path('./geojson')
print(f'Data directory: {DATA_DIR.resolve()}')

## 1. 抽出済みNER・里程構造化データ（CSV / JSON）の読み込みと統計確認

In [ ]:
df_prefs = pd.read_csv(DATA_DIR / 'taiping_huanyuji_prefectures.csv')
df_dists = pd.read_csv(DATA_DIR / 'taiping_huanyuji_distances.csv')
df_counties = pd.read_csv(DATA_DIR / 'taiping_huanyuji_counties.csv')
df_landmarks = pd.read_csv(DATA_DIR / 'taiping_huanyuji_landmarks.csv')

print(f'州府軍監数 (Prefectures): {len(df_prefs):,}')
print(f'四至八到里程エッジ数 (Distances): {len(df_dists):,}')
print(f'所属県数 (Counties): {len(df_counties):,}')
print(f'山川・古跡数 (Landmarks): {len(df_landmarks):,}')

display(df_prefs[['id', 'volume', 'dao', 'name', 'jun_name', 'kaiyuan_hu', 'song_total_hu', 'county_count', 'distance_count']].head(10))

## 2. 『四至八到』里程ネットワーク（NetworkX）の構築と宋代1里の実測距離較正

『太平寰宇記』の各州巻頭に記された「東至○州○里」「東南到○州○里」などの記述から有向グラフ $G = (V, E)$ を構築し、既知座標ペア間の測地線距離（Haversine距離）と記載里数の比から「1宋里 ≈ km」を算出します。

In [ ]:
DIRECTION_AZIMUTH_DEG = {
    '北': 0.0, '東北': 45.0, '東': 90.0, '東南': 135.0,
    '南': 180.0, '西南': 225.0, '西': 270.0, '西北': 315.0,
}

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2)**2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def offset_lonlat(lon, lat, azimuth_deg, dist_km):
    theta = math.radians(azimuth_deg)
    dlat = (dist_km * math.cos(theta)) / 111.1
    cos_lat = max(0.25, math.cos(math.radians(lat)))
    dlon = (dist_km * math.sin(theta)) / (111.1 * cos_lat)
    return lon + dlon, lat + dlat

# GeoJSONから座標付き州府データを読み込む
gdf_prefs = gpd.read_file(GEOJSON_DIR / 'prefectures.geojson')
gdf_edges = gpd.read_file(GEOJSON_DIR / 'distance_network.geojson')

G = nx.DiGraph()
for _, row in gdf_prefs.iterrows():
    G.add_node(row['name'], dao=row['dao'], lon=row.geometry.x, lat=row.geometry.y,
               song_hu=row['song_total_hu'], kaiyuan_hu=row['kaiyuan_hu'])

ratios = []
for _, edge in gdf_edges.iterrows():
    u, v = edge['source'], edge['target']
    li = edge['li']
    G.add_edge(u, v, direction=edge['direction'], li=li, geodesic_km=edge['geodesic_km'])
    if edge['geodesic_km'] and li and 20 <= li <= 2500:
        ratios.append(edge['geodesic_km'] / li)

LI_TO_KM = float(np.median(ratios))
print(f'NetworkX Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} directed edges')
print(f'実測較正値 (Median geodesic km per Song Li): 1里 = {LI_TO_KM:.4f} km')

## 3. 非線形最小二乗法による三辺測量（Trilateration）と残差検証

任意の州 $u$ について、隣接州 $v_i \in N(u)$ への記載里程 $d_i$（里）および方位角 $\theta_i$ から、以下の目的関数を最小化することで、『太平寰宇記』の里程記述のみから推定される三辺測量座標 $(\hat{\lambda}_u, \hat{\phi}_u)$ を求め、CHGIS 基準座標との空間残差（km）を検証します。

In [ ]:
def trilaterate_node(node_name, G, li_to_km=LI_TO_KM):
     anchors = []
    for _, tgt, edata in G.out_edges(node_name, data=True):
        if tgt in G.nodes and G.nodes[tgt].get('lon') is not None:
            anchors.append((G.nodes[tgt]['lon'], G.nodes[tgt]['lat'], edata['li'] * li_to_km, edata.get('direction', '')))
    if len(anchors) < 2:
        return None
    x0 = np.array([np.mean([a[0] for a in anchors]), np.mean([a[1] for a in anchors])])
    def residuals(pos):
        lon, lat = pos
        res = []
        for alon, alat, d_km, direction in anchors:
            dist = haversine_km(lon, lat, alon, alat)
            w = 1.0 / max(30.0, d_km)
            res.append((dist - d_km) * w)
            if direction in DIRECTION_AZIMUTH_DEG:
                opp_az = (DIRECTION_AZIMUTH_DEG[direction] + 180.0) % 360.0
                elon, elat = offset_lonlat(alon, alat, opp_az, d_km)
                res.append((lon - elon) * 35.0 * w)
                res.append((lat - elat) * 35.0 * w)
        return np.array(res)
    sol = least_squares(residuals, x0, loss='soft_l1', f_scale=1.0, max_nfev=200)
    est_lon, est_lat = float(sol.x[0]), float(sol.x[1])
    actual_lon, actual_lat = G.nodes[node_name]['lon'], G.nodes[node_name]['lat']
    err_km = haversine_km(actual_lon, actual_lat, est_lon, est_lat)
    return {'name': node_name, 'actual': (actual_lon, actual_lat), 'estimated': (est_lon, est_lat), 'error_km': round(err_km, 1), 'anchors': len(anchors)}

sample_results = [trilaterate_node(n, G) for n in ['開封府', '河南府', '京兆府', '太原府', '益州', '揚州', '洪州', '廣州']]
pd.DataFrame([r for r in sample_results if r])

## 4. Folium（Leaflet）によるインタラクティブ歴史地図の生成

十三道ポリゴン、387州府、四至八到ネットワーク、および山川・古跡レイヤーを重ね合わせた Folium インタラクティブ地図を生成します。

In [ ]:
gdf_dao = gpd.read_file(GEOJSON_DIR / 'dao_regions.geojson')

m = folium.Map(location=[34.2, 108.9], zoom_start=5, tiles='CartoDB dark_matter')

# 1. 十三道ポリゴンレイヤー
folium.GeoJson(
    gdf_dao,
    name='太平寰宇記・十三道',
    style_function=lambda feat: {
        'fillColor': feat['properties'].get('color', '#c9a962'),
        'color': feat['properties'].get('color', '#c9a962'),
        'weight': 1.8,
        'fillOpacity': 0.25,
    },
    tooltip=folium.GeoJsonTooltip(fields=['dao', 'prefecture_count', 'song_total_hu'], aliases=['道名', '州府数', '北宋戸数'])
).add_to(m)

# 2. 四至八到ネットワーク（主要エッジ）
fg_edges = folium.FeatureGroup(name='四至八到里程網（主要幹線）', show=False)
for _, edge in gdf_edges.head(500).iterrows():
    coords = [(lat, lon) for lon, lat in edge.geometry.coords]
    folium.PolyLine(coords, color='#f59e0b', weight=1.0, opacity=0.45,
                    tooltip=f"{edge['source']} →({edge['direction']} {edge['li']}里)→ {edge['target']}").add_to(fg_edges)
fg_edges.add_to(m)

# 3. 387州府軍監マーカー
fg_prefs = folium.FeatureGroup(name='州・府・軍・監 (387)', show=True)
for _, row in gdf_prefs.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4.5 if row['kind'] != '府' else 7.0,
        color='#fde68a' if row['kind'] == '府' else '#38bdf8',
        fill=True,
        fill_opacity=0.85,
        popup=f"<b>{row['name']}</b> ({row['dao']}・{row['volume']})<br>郡名: {row['jun_name']}<br>開元戸: {row['kaiyuan_hu']} / 北宋戸: {row['song_total_hu']}",
        tooltip=f"{row['name']} ({row['dao']})"
    ).add_to(fg_prefs)
fg_prefs.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m